# Практическая работа №1
## Классификация изображений: CNN и ResNet18

**Студентка:** Бахорина Ольга Александровна  
**Группа:** R4176c  
**Курс:** 1 курс магистратуры  
**Дисциплина:** «Прикладной искусственный интеллект»


## 1. Цель и постановка задачи

Цель работы — решить задачу многоклассовой классификации изображений
овощей и сравнить две модели:

1. собственную свёрточную нейронную сеть, обученную с нуля;
2. ResNet18 с предобученными весами ImageNet и последующим fine-tuning.

Подбор гиперпараметров выполняется только по validation-выборке.
Test-выборка используется после выбора конфигурации для итоговой оценки.


## 2. Данные и протокол эксперимента

Использован **Vegetable Image Dataset** для классификации изображений
овощей по 15 классам.

Разбиение данных:

| Выборка | Количество изображений |
|---|---:|
| Train | 14 996 |
| Validation | 3 000 |
| Test | 3 000 |

Исходное разбиение датасета сохраняется. Train используется для обучения,
validation — для выбора конфигурации и checkpoint, test — только для
финальной оценки.

Основные метрики: **accuracy** и **macro-F1**.

Полное обучение выполнено на Kaggle GPU T4×2. Для воспроизводимости
используется `seed = 42`.


## 3. Подготовка окружения


In [ ]:
import datetime
import hashlib
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path, PurePosixPath

import torch

EXPECTED_COMMIT = "b0daa7d026774ad899168822a2b21045002a1e72"
EXPECTED_SOURCE_ARCHIVE_SHA256 = (
    "c71bce8c0bf650a881e62eab41173a12ef77843d17e5f3a48f2fdf2ddfebeb86"
)

INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working/project")
OUTPUT_ROOT = Path("/kaggle/working/bridge-output")


def input_mount(slug: str) -> Path:
    """Return the single Kaggle input directory matching ``slug``."""
    candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / "datasets" / "senatorov1" / slug,
    ]
    matches = [path for path in candidates if path.is_dir()]

    if len(matches) != 1:
        raise RuntimeError(
            "Expected one mounted private input for "
            f"{slug}, found {matches}"
        )

    return matches[0]


SOURCE_MOUNT = input_mount("aai-p01-source-b0daa7d02677")
DATA = input_mount("aai-p01-vegetable-f82f6d8ab7b")

SOURCE_ARCHIVE = SOURCE_MOUNT / "source.tar.gz"
SOURCE_TREE = SOURCE_MOUNT / "source"


In [ ]:
EXPECTED_SOURCE_HASHES = {
    "aai/__init__.py": (
            "e4dc8708e5ff537e14ffca4539398f2edc3b19624da66de4e8bfbe76fdddc66c"
    ),
    "aai/__main__.py": (
            "62f0f08bf5d2d3b2013fea12a31374dbce1a8608231605ffd7c3796b6ef9034f"
    ),
    "aai/common/__init__.py": (
            "3a4654a33671038d1abd3bfc4e5eb9d44542a78791e87c45aadab337e92c716a"
    ),
    "aai/common/acquisition.py": (
            "3e325edcc6ace7bb8e097a2d01bfa5efc03ab8d52f080852ad0a4076c20e55a5"
    ),
    "aai/common/artifacts.py": (
            "7da52c9a317c20efadd057143afe414fae162365130e75614ed98a5515c308af"
    ),
    "aai/common/audit_checks.py": (
            "2967424f57b9769d5ee186ad97cc447aae795da9b0391184d1a44fcbbc22a2d2"
    ),
    "aai/common/config.py": (
            "9d35975289d17a6e8932e87093ee305b6ced500e1c62b54e78c50138f952d764"
    ),
    "aai/common/data.py": (
            "b56c6d9a49c544bf442ae3e8514bcf88fe526b6daff1850c1873c160f0b2fc10"
    ),
    "aai/common/kaggle.py": (
            "7688cb6a4c9d9afefebb7196d96734b2d13f58729e740d5e56e18b7c9758e5a0"
    ),
    "aai/common/perceptual_audit.py": (
            "7223b999d5a3eca83866c77fb0e666a4cb1814de428bfb2b593c130de190d553"
    ),
    "aai/common/reporting.py": (
            "4b68580b89b8d03570d4344c5e012bb672d773141f1d256157057e2b967e4205"
    ),
    "aai/common/runtime.py": (
            "41c9aadba4edfbdb7a9fdd3fb24ae0481cb9de8172dcb97352e0e1b3f8d467ca"
    ),
    "aai/common/services.py": (
            "407576006bc8545e97e7016be69e696e06f084b1744c9b4e56b456950ad03c62"
    ),
    "aai/common/supervised.py": (
            "1a482bece5a7c27391da9a6e69a897ed55eb98b979b6250ed8738cd44aac5ef2"
    ),
    "aai/p1/__init__.py": (
            "b1d8a8d84c96eff1cd563dd33d55c90bf72b0daceb93f62dd002fdabdefed653"
    ),
    "aai/p1/data.py": (
            "97c1ccedb19db4b760cc08ec49ba277e88078fa4b6e3ae0419f4a6865d17a16a"
    ),
    "aai/p1/experiment.py": (
            "735b67033f13321aa6f6a27e6dd95718298a91afc57b58156f8465a0cede8e2e"
    ),
    "aai/p1/kaggle.py": (
            "4ad50007595dae726522fa4d037c58e59464771ca35ce5392cf655f6c7b44547"
    ),
    "aai/p1/models.py": (
            "6e1c41b9ed841bbe586194cde027fac6ef94c5df22ff958bd197c32499437aaf"
    ),
    "aai/p2/__init__.py": (
            "c6f488ada628b0d7c8875495b957bfa77ed8257dc2394ed36ec491833500724f"
    ),
    "aai/p2/audit.py": (
            "55aee548c58e4137aafa7c9098489f655dcbabb7425c2e9ecc4868c7f7cd1973"
    ),
    "aai/p2/data.py": (
            "8b2bcae62708caf95d43a0d2bb2694e0bbdbfb5a13de0aa1996b1da07395ec32"
    ),
    "aai/p2/experiment.py": (
            "80e73e797426df3d353797f55c999585971e9bf30890007ac175f9d27bff7410"
    ),
    "aai/p2/groups.py": (
            "4f3663f7ae46c8df3e8a1c54f8fe7fe951e8c7d19886f5201e7e9914c51aad7f"
    ),
    "aai/p2/kaggle.py": (
            "daddb5d6c03784e29ac845bad13a6c966d6ac7064dc57f46f0c6658c2ad5538f"
    ),
    "aai/p2/metrics.py": (
            "1438dbef0b822cbf5b6cf9e9cc3ff6110c3fcc1a529a45fb994c1f40c1f22faa"
    ),
    "aai/p2/models.py": (
            "d230ff600809211882c5fddecc73db2411122a27a69af5cb183b61bbe6fdb6a2"
    ),
    "aai/p2/revision.py": (
            "f9ff26b4a82996d249e03a212bf3f059bd1f68b9a253368cfcc957f4e216e6ac"
    ),
    "aai/p2/yolo_view.py": (
            "04e982ddb583b0df322de4c83f3bcb92150ee02a4d53c0d7eb1363c8af03d907"
    ),
    "aai/p3/__init__.py": (
            "0fa72008d56c75c7f73c26d8b35c1073e904ea21c62ac30fbad5328a3a6bf4f8"
    ),
    "aai/p3/data.py": (
            "7dd96ae002ba7691a9d86343d2b7531cbb2b4f48328280997985da027c5b6b3c"
    ),
    "aai/p3/experiment.py": (
            "980bed925c4939c369cee8b2aa0948c7f548a42ae1aeef6d5938bb2683489c80"
    ),
    "aai/p3/kaggle.py": (
            "6acb2773a7468feeca120a3582bee2a02e67eb40edef75f0a9ce662cd16d901d"
    ),
    "aai/p3/models.py": (
            "00dc3dc0a3ade8c09ff5ba2d875b4b3703644b844d825c89688686b945a6d1fc"
    ),
    "aai/p4/__init__.py": (
            "310ff0844c36bf3f8a732de15d3698ece16d7264f9a19783d2f490e54d43be40"
    ),
    "aai/p4/data.py": (
            "8a413a1ed24dbf9ee717648a419ba5ecfec35f86e7d4e1c8088d08b35ee0ef4d"
    ),
    "aai/p4/experiment.py": (
            "62d67806922969da86446f301bc3721a2990463a192b0b455b9841cea3020bbd"
    ),
    "aai/p4/metrics.py": (
            "9dbd8e0f03f3940ae4bc03615806a5c833d882705ecc563b073facfa68e5e3fb"
    ),
    "aai/p4/models.py": (
            "d644dd1a1507bd80dcbb35a1c91c0d5b1fe262bd0dfbf967a21dd158e0b3cbd4"
    ),
    "practice_01/configs/budget.yaml": (
            "0037ceffe4d7104deb21eb02e01fb0e802609ef0ebfe59b464590679168ba08c"
    ),
    "practice_01/configs/cnn_a.yaml": (
            "af20181629f0634935bba7110da4e6479dc3e307111f7231f3be6bb31ee078d2"
    ),
    "practice_01/configs/cnn_b.yaml": (
            "2dd37e441f9fb8fa7f33db914d93f8ba99c0e0e324f7e4becbe0bd6c4d9fbec1"
    ),
    "practice_01/configs/resnet_a.yaml": (
            "65d42127bbce0850d8b1578834b5bb806a86df46f77d9283e0da0f9e78fa10a8"
    ),
    "practice_01/configs/resnet_b.yaml": (
            "598700c6f2192d1a32acf080ea6f4578205ae6d84dfa55984cc7d47839b97efd"
    ),
    "preflight.evidence.json": (
            "34402628671acc79a4613ec5f26f13ff92e2d7e541d195f992fc0b2c142298eb"
    ),
}


In [ ]:
def _safe_target(root: Path, relative: PurePosixPath) -> Path:
    """Resolve a source member and reject path traversal."""
    target = (root / Path(*relative.parts)).resolve()
    root_resolved = root.resolve()

    if (
        relative.is_absolute()
        or ".." in relative.parts
        or root_resolved not in target.parents
    ):
        raise RuntimeError(f"Unsafe source member: {relative}")

    return target


def _extract_source_archive() -> None:
    """Verify and extract the pinned source archive into ``WORK``."""
    archive_sha256 = hashlib.sha256(
        SOURCE_ARCHIVE.read_bytes()
    ).hexdigest()

    if archive_sha256 != EXPECTED_SOURCE_ARCHIVE_SHA256:
        raise RuntimeError("Pinned source package hash mismatch")

    WORK.mkdir(parents=True, exist_ok=False)
    seen_hashes = {}

    with tarfile.open(SOURCE_ARCHIVE, "r:gz") as archive:
        for member in archive.getmembers():
            relative = PurePosixPath(member.name)
            target = _safe_target(WORK, relative)

            if (
                not member.isfile()
                or member.issym()
                or member.islnk()
                or member.name in seen_hashes
            ):
                raise RuntimeError(
                    f"Unsafe source member: {member.name}"
                )

            stream = archive.extractfile(member)
            if stream is None:
                raise RuntimeError(
                    f"Unreadable source member: {member.name}"
                )

            content = stream.read()
            seen_hashes[member.name] = hashlib.sha256(
                content
            ).hexdigest()

            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)

    if seen_hashes != EXPECTED_SOURCE_HASHES:
        raise RuntimeError("Pinned source tree manifest mismatch")


def _copy_source_tree() -> None:
    """Verify and copy an already extracted pinned source tree."""
    actual_hashes = {}

    for path in sorted(SOURCE_TREE.rglob("*")):
        if path.is_symlink():
            raise RuntimeError(
                "Pinned source tree contains a symlink"
            )

        if path.is_file():
            relative_name = path.relative_to(
                SOURCE_TREE
            ).as_posix()
            actual_hashes[relative_name] = hashlib.sha256(
                path.read_bytes()
            ).hexdigest()

    if actual_hashes != EXPECTED_SOURCE_HASHES:
        raise RuntimeError("Pinned extracted source tree mismatch")

    WORK.mkdir(parents=True, exist_ok=False)

    for name in EXPECTED_SOURCE_HASHES:
        relative = PurePosixPath(name)
        target = _safe_target(WORK, relative)
        source = SOURCE_TREE.joinpath(*relative.parts)

        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)


if SOURCE_ARCHIVE.is_file():
    _extract_source_archive()
elif SOURCE_TREE.is_dir():
    _copy_source_tree()
else:
    raise RuntimeError(
        "Missing pinned source package: "
        f"{SOURCE_ARCHIVE} or {SOURCE_TREE}"
    )

sys.path.insert(0, str(WORK))


In [ ]:
from aai.common.artifacts import load_checkpoint, read_json
from aai.common.kaggle import (
    acquire_approved_downloads,
    approved_downloads_from_configs,
    install_dataset_contracts,
    validate_approved_downloads,
    validate_embedded_preflight,
    validate_resume,
)


In [ ]:
manifest = {
    "approved_downloads": [
        {
            "cache_relpath": "hub/checkpoints/resnet18-f37072fd.pth",
            "sha256": (
                    "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec"
            ),
            "url": "https://download.pytorch.org/models/resnet18-f37072fd.pth",
        },
    ],
    "commit": "b0daa7d026774ad899168822a2b21045002a1e72",
    "config": {
        "paths": [
            "practice_01/configs/cnn_a.yaml",
            "practice_01/configs/cnn_b.yaml",
            "practice_01/configs/resnet_a.yaml",
            "practice_01/configs/resnet_b.yaml",
        ],
        "resolved": {
            "practice_01/configs/cnn_a.yaml": {
                "budget": {
                    "file": "practice_01/configs/budget.yaml",
                },
                "data": {
                    "dataset": "vegetable",
                    "prepared_revision": "current",
                },
                "experiment_id": "p1_cnn_a",
                "mode": "full",
                "model": {
                    "channels": [
                        32,
                        64,
                        128,
                    ],
                    "dropout": 0.2,
                    "name": "custom_cnn",
                    "num_classes": 15,
                    "weights_path": None,
                },
                "paths": {
                    "artifact_root": "practice_01/artifacts",
                    "data_root": "data",
                },
                "practice": 1,
                "runtime": {
                    "device": "auto",
                    "num_workers": 0,
                    "precision": "fp32",
                    "seed": 42,
                },
                "schema_version": 1,
                "train": {
                    "batch_size_start": 32,
                    "checkpoint_direction": "max",
                    "checkpoint_metric": "val_accuracy",
                    "lr": 0.001,
                    "optimizer": "adam",
                    "scheduler": "none",
                    "weight_decay": 0.0001,
                },
            },
            "practice_01/configs/cnn_b.yaml": {
                "budget": {
                    "file": "practice_01/configs/budget.yaml",
                },
                "data": {
                    "dataset": "vegetable",
                    "prepared_revision": "current",
                },
                "experiment_id": "p1_cnn_b",
                "mode": "full",
                "model": {
                    "channels": [
                        32,
                        64,
                        128,
                    ],
                    "dropout": 0.2,
                    "name": "custom_cnn",
                    "num_classes": 15,
                    "weights_path": None,
                },
                "paths": {
                    "artifact_root": "practice_01/artifacts",
                    "data_root": "data",
                },
                "practice": 1,
                "runtime": {
                    "device": "auto",
                    "num_workers": 0,
                    "precision": "fp32",
                    "seed": 42,
                },
                "schema_version": 1,
                "train": {
                    "batch_size_start": 32,
                    "checkpoint_direction": "max",
                    "checkpoint_metric": "val_accuracy",
                    "lr": 0.0003,
                    "optimizer": "adam",
                    "scheduler": "none",
                    "weight_decay": 0.0001,
                },
            },
            "practice_01/configs/resnet_a.yaml": {
                "budget": {
                    "file": "practice_01/configs/budget.yaml",
                },
                "data": {
                    "dataset": "vegetable",
                    "prepared_revision": "current",
                },
                "experiment_id": "p1_resnet_a",
                "mode": "full",
                "model": {
                    "channels": [
                        32,
                        64,
                        128,
                    ],
                    "dropout": 0.2,
                    "name": "resnet18",
                    "num_classes": 15,
                    "weights_path": None,
                    "weights_sha256": (
                            "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c"
                            "560e07ec"
                    ),
                    "weights_source": (
                            "https://download.pytorch.org/models/resnet18-f37072fd.pt"
                            "h"
                    ),
                    "weights_url": (
                            "https://download.pytorch.org/models/resnet18-f37072fd.pt"
                            "h"
                    ),
                },
                "paths": {
                    "artifact_root": "practice_01/artifacts",
                    "data_root": "data",
                },
                "practice": 1,
                "runtime": {
                    "device": "auto",
                    "num_workers": 0,
                    "precision": "fp32",
                    "seed": 42,
                },
                "schema_version": 1,
                "train": {
                    "batch_size_start": 32,
                    "checkpoint_direction": "max",
                    "checkpoint_metric": "val_accuracy",
                    "finetune_backbone_lr": 0.0001,
                    "finetune_head_lr": 0.001,
                    "lr": 0.001,
                    "optimizer": "adam",
                    "scheduler": "none",
                    "weight_decay": 0.0001,
                },
            },
            "practice_01/configs/resnet_b.yaml": {
                "budget": {
                    "file": "practice_01/configs/budget.yaml",
                },
                "data": {
                    "dataset": "vegetable",
                    "prepared_revision": "current",
                },
                "experiment_id": "p1_resnet_b",
                "mode": "full",
                "model": {
                    "channels": [
                        32,
                        64,
                        128,
                    ],
                    "dropout": 0.2,
                    "name": "resnet18",
                    "num_classes": 15,
                    "weights_path": None,
                    "weights_sha256": (
                            "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c"
                            "560e07ec"
                    ),
                    "weights_source": (
                            "https://download.pytorch.org/models/resnet18-f37072fd.pt"
                            "h"
                    ),
                    "weights_url": (
                            "https://download.pytorch.org/models/resnet18-f37072fd.pt"
                            "h"
                    ),
                },
                "paths": {
                    "artifact_root": "practice_01/artifacts",
                    "data_root": "data",
                },
                "practice": 1,
                "runtime": {
                    "device": "auto",
                    "num_workers": 0,
                    "precision": "fp32",
                    "seed": 42,
                },
                "schema_version": 1,
                "train": {
                    "batch_size_start": 32,
                    "checkpoint_direction": "max",
                    "checkpoint_metric": "val_accuracy",
                    "finetune_backbone_lr": 3e-05,
                    "finetune_head_lr": 0.0003,
                    "lr": 0.001,
                    "optimizer": "adam",
                    "scheduler": "none",
                    "weight_decay": 0.0001,
                },
            },
        },
        "sha256": "4828a3d3a94eef51a0c84f5c5268b26b65a318b27a115efe528469cfb56d82e9",
    },
    "dataset_contract": {
        "audit_status": "passed",
        "dataset_id": (
                "f82f6d8ab7b64c9481bdb94d436cab25f0ecc0998302850b406b23e56645e5ed"
        ),
        "files": [
            "audit.json",
            "audit_checks.json",
            "dataset.json",
            "excluded.jsonl",
            "manifest.jsonl",
            "manifest_before_cleanup.jsonl",
            "manifest_test.jsonl",
            "manifest_train.jsonl",
            "manifest_val.jsonl",
            "mapping.json",
            "source.json",
        ],
        "hashes": {
            "audit.json": (
                    "941bcbd3530b5d940416a4254d3fe1967df5a4e8e35e428c2db807405bce0d0d"
            ),
            "audit_checks.json": (
                    "2f0cef51b3dfd3aa5f89b0c9d57ba742a93a827cefa4b398a2bb6dd989c5ab87"
            ),
            "dataset.json": (
                    "a12b2575f25ebff344cde34546bd0eb939b62d080e656c804e7d29bd85882366"
            ),
            "excluded.jsonl": (
                    "113910772edd46698d5e9ef2ef5f33c62fdea51998b34b5c1b64854d49f2384e"
            ),
            "manifest.jsonl": (
                    "543732f9987faa4caed818a0bed11785b9add4c440a805a53373c876b372f45d"
            ),
            "manifest_before_cleanup.jsonl": (
                    "236b2a3a822907a20d2c23f7f97a4cb1aca3eccab3a7c7571034aac92413feb0"
            ),
            "manifest_test.jsonl": (
                    "b6d114f1e436a53e4b68240eaced337bf25906cb9ef4fed90e16e5b10beb224b"
            ),
            "manifest_train.jsonl": (
                    "11644e372a4a74c7f1d227a4d197c28535a70b96c17320b77664d47722db081c"
            ),
            "manifest_val.jsonl": (
                    "4952cb625f574af02a515523aaa166dfdb25dd4f962db3f9e27fad15dd039d30"
            ),
            "mapping.json": (
                    "f66b97cb34b7eb7415745d933b22833c5ac03159e3d964288e326baed77308b1"
            ),
            "source.json": (
                    "9d286a9ee3c26fc0643edcfe229840f81870193bcfc672dc953e2e4e492ff52c"
            ),
        },
        "practice": 1,
        "revision": "current",
        "schema_version": 1,
    },
    "dataset_id": "f82f6d8ab7b64c9481bdb94d436cab25f0ecc0998302850b406b23e56645e5ed",
    "environment": {
        "accelerator": "gpu",
        "disk": None,
        "gpu_quota": None,
        "internet": True,
        "memory_bytes": None,
        "platform": "kaggle",
        "session_limits": None,
    },
    "execution_contract": [
        "preflight",
        "resume_compatibility",
        "full",
    ],
    "hashes": {
    },
    "mode": "full",
    "owner": "senatorov1",
    "practice": 1,
    "preflight_evidence_sha256": (
            "41aab22d974d27b2ddd93170bfdac9017a707497b4926cb690df7a4af97050aa"
    ),
    "private": True,
    "run_id": "p01-full-v1-b0daa7d02677-f82f6d8a",
    "schema_version": 1,
    "seed": 42,
    "slug": "aai-p01-cnn-resnet18-kaggle",
    "status": "planned",
    "timestamps": {
        "built_at": "2026-09-14T03:24:00Z",
        "completed_at": None,
        "started_at": None,
        "submitted_at": None,
    },
    "version": 1,
}


## 4. Обучение моделей

Проверены четыре конфигурации:

| Конфигурация | Модель | Learning rate |
|---|---|---:|
| `p1_cnn_a` | CNN | 0.001 |
| `p1_cnn_b` | CNN | 0.0003 |
| `p1_resnet_a` | ResNet18 | 0.001 |
| `p1_resnet_b` | ResNet18 | 0.001 |

Для всех конфигураций используется Adam, `weight_decay = 1e-4`
и начальный `batch_size = 32`.

Для ResNet18 отдельно задаются learning rate классификатора и
размораживаемой части backbone. Начальные веса ResNet18 загружаются
из официального checkpoint PyTorch.


In [ ]:
DDP_PRACTICES = {1, 3}

ddp_enabled = manifest["practice"] in DDP_PRACTICES
requested_world = (
    int(os.environ.get("AAI_KAGGLE_DDP_WORLD_SIZE", "2"))
    if ddp_enabled
    else 1
)

if requested_world < 1:
    raise RuntimeError(
        "AAI_KAGGLE_DDP_WORLD_SIZE must be positive"
    )

available_world = (
    torch.cuda.device_count() if torch.cuda.is_available() else 0
)

if ddp_enabled and available_world < requested_world:
    raise RuntimeError(
        f"Requested {requested_world} GPUs, "
        f"but only {available_world} are visible"
    )

distributed_world = requested_world if ddp_enabled else 1


def distributed_command(command: str) -> str:
    """Wrap an ``aai`` command in torch.distributed when needed."""
    if distributed_world <= 1:
        return command

    parts = shlex.split(command)
    if len(parts) >= 3 and parts[:3] == ["python", "-m", "aai"]:
        distributed_prefix = [
            "python",
            "-m",
            "torch.distributed.run",
            "--standalone",
            "--nproc_per_node",
            str(distributed_world),
            "-m",
            "aai",
        ]
        return " ".join(distributed_prefix + parts[3:])

    return command


In [ ]:
gpu_result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
    check=False,
)

evidence = read_json(WORK / "preflight.evidence.json")

page_size = os.sysconf("SC_PAGE_SIZE")
total_pages = os.sysconf("SC_PHYS_PAGES")
available_pages = os.sysconf("SC_AVPHYS_PAGES")

preflight = {
    "schema_version": 1,
    "gpu": {
        "available": (
            gpu_result.returncode == 0
            and bool(gpu_result.stdout.strip())
        ),
        "description": gpu_result.stdout.strip(),
    },
    "memory_bytes": {
        "total": page_size * total_pages,
        "available": page_size * available_pages,
    },
    "disk_bytes": shutil.disk_usage(
        "/kaggle/working"
    )._asdict(),
    "internet_enabled": bool(evidence["internet_enabled"]),
    "session": {
        "run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
    },
}

validate_embedded_preflight(
    preflight,
    evidence,
    manifest["preflight_evidence_sha256"],
    approved_downloads=manifest["approved_downloads"],
)

config_paths = [
    WORK / path for path in manifest["config"]["paths"]
]
tracked_downloads = approved_downloads_from_configs(config_paths)

validate_approved_downloads(
    manifest["approved_downloads"],
    tracked_downloads,
)

install_dataset_contracts(
    DATA,
    WORK,
    manifest["practice"],
    manifest["dataset_contract"],
)

cache_root = Path(
    os.environ.get(
        "TORCH_HOME",
        Path.home() / ".cache" / "torch",
    )
)

acquire_approved_downloads(
    manifest["approved_downloads"],
    cache_root,
    allow_network=preflight["internet_enabled"],
)


In [1]:
base_commands = [
    (
        "python -m aai train --practice 1 "
        "--config practice_01/configs/cnn_a.yaml"
    ),
    (
        "python -m aai train --practice 1 "
        "--config practice_01/configs/cnn_b.yaml"
    ),
    (
        "python -m aai train --practice 1 "
        "--config practice_01/configs/resnet_a.yaml"
    ),
    (
        "python -m aai train --practice 1 "
        "--config practice_01/configs/resnet_b.yaml"
    ),
]

commands = [
    distributed_command(f"{command} --data-root {DATA}")
    for command in base_commands
]

resume_run_dir = os.environ.get("AAI_RUN_DIR")
resume_checkpoint = os.environ.get("AAI_CHECKPOINT")

if bool(resume_run_dir) != bool(resume_checkpoint):
    raise RuntimeError(
        "Resume requires both AAI_RUN_DIR and AAI_CHECKPOINT"
    )

if resume_run_dir:
    run_dir = Path(resume_run_dir)
    validate_resume(
        read_json(run_dir / "run.json"),
        load_checkpoint(resume_checkpoint),
    )
    commands = [
        distributed_command(
            "python -m aai resume --practice 1 "
            f"--run-dir {run_dir} --data-root {DATA}"
        )
    ]

for command in commands:
    subprocess.run(
        shlex.split(command),
        cwd=WORK,
        check=True,
    )


/kaggle/working/project/practice_01/artifacts/runs/20260914T033444204250Z_p1_cnn_a_fce25d5e
/kaggle/working/project/practice_01/artifacts/runs/20260914T034345350159Z_p1_cnn_b_34087241
/kaggle/working/project/practice_01/artifacts/runs/20260914T035210722298Z_p1_resnet_a_d0a47ce7
/kaggle/working/project/practice_01/artifacts/runs/20260914T040100662208Z_p1_resnet_b_91a4c47d


In [ ]:
artifact_root = (
    WORK
    / f"practice_{manifest['practice']:02d}"
    / "artifacts"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

artifact_hashes = {}

artifact_files = sorted(
    path
    for path in artifact_root.rglob("*")
    if path.is_file() and not path.is_symlink()
)

for path in artifact_files:
    relative = path.relative_to(artifact_root)
    target = OUTPUT_ROOT / relative

    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(path, target)

    artifact_hashes[relative.as_posix()] = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

if not artifact_hashes:
    raise RuntimeError("Kaggle run produced no artifact files")

manifest["hashes"] = artifact_hashes
manifest["status"] = "completed"

manifest["environment"].update(
    {
        "python": sys.version,
        "os": platform.platform(),
        "accelerator": gpu_result.stdout.strip(),
        "distributed_world_size": distributed_world,
        "memory_bytes": page_size * total_pages,
        "disk": shutil.disk_usage(
            "/kaggle/working"
        )._asdict(),
        "session_limits": {
            "kernel_run_type": os.environ.get(
                "KAGGLE_KERNEL_RUN_TYPE"
            )
        },
        "gpu_quota": {
            "status": "not_exposed_to_kernel"
        },
    }
)

completed_at = datetime.datetime.now(
    datetime.timezone.utc
).isoformat()
manifest["timestamps"]["completed_at"] = completed_at.replace(
    "+00:00",
    "Z",
)

provenance_path = OUTPUT_ROOT / "provenance.json"
provenance_text = json.dumps(
    manifest,
    sort_keys=True,
    indent=2,
)
provenance_path.write_text(
    provenance_text + "\n",
    encoding="utf-8",
)


## 5. Результаты

По validation выбраны `p1_cnn_a` и `p1_resnet_a`.
Test-выборка при выборе конфигураций не использовалась.

| Модель | Val accuracy | Val macro-F1 | Test accuracy | Test macro-F1 | Test loss |
|---|---:|---:|---:|---:|---:|
| CNN | 0.936333 | 0.936319 | 0.937667 | 0.937451 | 0.226347 |
| ResNet18 | 0.999667 | 0.999667 | 0.999000 | 0.998999 | 0.004158 |

Проверка сохранённых checkpoint завершена успешно.


In [ ]:
selected = {
    "cnn": {
        "experiment_id": "p1_cnn_a",
        "run_id": "20260914T033444204250Z_p1_cnn_a_fce25d5e",
        "val_accuracy": 0.9363333333333334,
        "val_macro_f1": 0.9363187336642439,
        "test_accuracy": 0.9376666666666666,
        "test_macro_f1": 0.9374514356959985,
        "test_loss": 0.2263465506931146,
    },
    "resnet18": {
        "experiment_id": "p1_resnet_a",
        "run_id": "20260914T035210722298Z_p1_resnet_a_d0a47ce7",
        "val_accuracy": 0.9996666666666667,
        "val_macro_f1": 0.9996666645833202,
        "test_accuracy": 0.999,
        "test_macro_f1": 0.998999154129973,
        "test_loss": 0.004157620379325349,
    },
}
checkpoint_verification = "verified; 4 passed"
selected


### 5.1. Динамика обучения

Ниже строятся графики по сохранённым `history.csv` для выполненных
экспериментов.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

history_files = sorted(
    artifact_root.rglob("history.csv")
)
print("Найдено history.csv:", len(history_files))

for history_path in history_files:
    history = pd.read_csv(history_path)
    run_dir = history_path.parent
    run_name = run_dir.name

    display(Markdown(f"### `{run_name}`"))
    display(history)

    metric_pairs = [
        (
            "train_accuracy",
            "val_accuracy",
            "Accuracy",
        ),
        (
            "train_loss",
            "val_loss",
            "Loss",
        ),
        (
            "train_macro_f1",
            "val_macro_f1",
            "Macro-F1",
        ),
        (
            "train_f1_macro",
            "val_f1_macro",
            "Macro-F1",
        ),
    ]

    for train_col, val_col, title in metric_pairs:
        if (
            train_col not in history.columns
            and val_col not in history.columns
        ):
            continue

        plt.figure(figsize=(8, 4))

        if train_col in history.columns:
            plt.plot(
                range(1, len(history) + 1),
                history[train_col],
                marker="o",
                label=train_col,
            )

        if val_col in history.columns:
            plt.plot(
                range(1, len(history) + 1),
                history[val_col],
                marker="o",
                label=val_col,
            )

        plt.xlabel("Эпоха")
        plt.ylabel(title)
        plt.title(f"{run_name}: {title}")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()


### 5.2. Сравнение итоговых метрик


In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    {
        "model": "Custom CNN",
        "val_accuracy": selected["cnn"]["val_accuracy"],
        "val_macro_f1": selected["cnn"]["val_macro_f1"],
        "test_accuracy": selected["cnn"]["test_accuracy"],
        "test_macro_f1": selected["cnn"]["test_macro_f1"],
        "test_loss": selected["cnn"]["test_loss"],
    },
    {
        "model": "ResNet18",
        "val_accuracy": selected["resnet18"]["val_accuracy"],
        "val_macro_f1": selected["resnet18"]["val_macro_f1"],
        "test_accuracy": selected["resnet18"]["test_accuracy"],
        "test_macro_f1": selected["resnet18"]["test_macro_f1"],
        "test_loss": selected["resnet18"]["test_loss"],
    },
]).set_index("model")

display(results_df)

ax = results_df[["val_accuracy", "test_accuracy"]].plot(
    kind="bar", figsize=(8, 4), ylim=(0.85, 1.01), rot=0
)
ax.set_ylabel("Accuracy")
ax.set_title("Validation и test accuracy")
ax.grid(axis="y", alpha=0.3)
plt.show()


### 5.3. Детальная оценка на test

Для выбранных CNN и ResNet18 используются сохранённые
test-предсказания. Если prediction-артефакты ещё не созданы,
выполняется финальная оценка выбранных checkpoint на test.

Далее рассчитываются:

- confusion matrix;
- precision, recall и F1-score по каждому классу;
- macro- и weighted-средние;
- примеры ошибочных классификаций;
- примеры правильных классификаций.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)


def run_test_evaluation() -> None:
    """Create test prediction artifacts when they are absent."""
    selection_path = artifact_root / "selection.json"

    if not selection_path.exists():
        select_command = [
            sys.executable,
            "-m",
            "aai",
            "select",
            "--practice",
            "1",
            "--artifact-root",
            str(artifact_root),
        ]
        subprocess.run(
            select_command,
            cwd=WORK,
            check=True,
        )

    evaluate_command = [
        sys.executable,
        "-m",
        "aai",
        "evaluate",
        "--practice",
        "1",
        "--selection",
        str(selection_path),
        "--split",
        "test",
        "--data-root",
        str(DATA),
        "--device",
        "cuda:0",
    ]

    subprocess.run(
        evaluate_command,
        cwd=WORK,
        check=True,
    )


def read_prediction_file(path: Path) -> pd.DataFrame:
    """Read a prediction artifact into a DataFrame."""
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".jsonl":
        records = []
        with path.open(
            "r",
            encoding="utf-8",
        ) as stream:
            for line in stream:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.DataFrame(records)

    if suffix == ".json":
        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        if isinstance(data, list):
            return pd.DataFrame(data)

        if isinstance(data, dict):
            for key in (
                "predictions",
                "rows",
                "records",
                "data",
            ):
                value = data.get(key)
                if isinstance(value, list):
                    return pd.DataFrame(value)

    return pd.DataFrame()


TRUE_COLUMNS = (
    "y_true",
    "true_label",
    "target",
    "target_id",
    "actual",
    "ground_truth",
    "label",
)

PRED_COLUMNS = (
    "y_pred",
    "pred_label",
    "prediction",
    "prediction_id",
    "predicted",
    "predicted_label",
    "pred_class",
)

PATH_COLUMNS = (
    "image_path",
    "path",
    "filepath",
    "file",
    "filename",
    "image",
)

CONFIDENCE_COLUMNS = (
    "confidence",
    "probability",
    "pred_probability",
    "score",
)


def first_existing_column(
    frame: pd.DataFrame,
    candidates: tuple[str, ...],
) -> str | None:
    """Return the first matching column name."""
    lowered = {
        str(column).lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]

    return None


In [ ]:
def prediction_candidates() -> list[Path]:
    """Return likely prediction artifacts, ranked by relevance."""
    roots = [artifact_root]

    if OUTPUT_ROOT.exists():
        roots.append(OUTPUT_ROOT)

    candidates = []

    for root in roots:
        for path in root.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower()
                in {".csv", ".json", ".jsonl"}
            ):
                name = path.as_posix().lower()

                if (
                    "pred" in name
                    or "test" in name
                    or "evaluation" in name
                ):
                    candidates.append(path)

    def score(path: Path) -> tuple[int, str]:
        name = path.as_posix().lower()
        value = 0

        if "test" in name:
            value += 4
        if "pred" in name:
            value += 3
        if "evaluation" in name:
            value += 2

        return (-value, name)

    return sorted(
        set(candidates),
        key=score,
    )


def normalize_prediction_table(
    frame: pd.DataFrame,
) -> pd.DataFrame | None:
    """Normalize true/pred/path/confidence column names."""
    if frame.empty:
        return None

    split_column = first_existing_column(
        frame,
        ("split", "dataset_split"),
    )

    if split_column is not None:
        test_mask = (
            frame[split_column]
            .astype(str)
            .str.lower()
            .eq("test")
        )
        if test_mask.any():
            frame = frame.loc[test_mask].copy()

    true_column = first_existing_column(
        frame,
        TRUE_COLUMNS,
    )
    pred_column = first_existing_column(
        frame,
        PRED_COLUMNS,
    )

    if (
        true_column is None
        or pred_column is None
    ):
        return None

    normalized = frame.copy()
    normalized = normalized.rename(
        columns={
            true_column: "y_true",
            pred_column: "y_pred",
        }
    )

    path_column = first_existing_column(
        normalized,
        PATH_COLUMNS,
    )
    if path_column is not None:
        normalized = normalized.rename(
            columns={
                path_column: "image_path"
            }
        )

    confidence_column = first_existing_column(
        normalized,
        CONFIDENCE_COLUMNS,
    )
    if confidence_column is not None:
        normalized = normalized.rename(
            columns={
                confidence_column: "confidence"
            }
        )

    probability_columns = [
        column
        for column in normalized.columns
        if str(column).lower().startswith(
            ("prob_", "probability_")
        )
    ]

    if (
        "confidence" not in normalized.columns
        and probability_columns
    ):
        normalized["confidence"] = (
            normalized[
                probability_columns
            ]
            .astype(float)
            .max(axis=1)
        )

    return normalized.reset_index(drop=True)


def load_predictions(
    run_id: str,
    experiment_id: str,
) -> pd.DataFrame:
    """Load test predictions for one selected model."""
    all_candidates = prediction_candidates()

    preferred = [
        path
        for path in all_candidates
        if (
            run_id in path.as_posix()
            or experiment_id
            in path.as_posix()
        )
    ]

    search_order = preferred + [
        path
        for path in all_candidates
        if path not in preferred
    ]

    for path in search_order:
        try:
            frame = read_prediction_file(path)
        except (
            ValueError,
            json.JSONDecodeError,
            UnicodeDecodeError,
        ):
            continue

        normalized = normalize_prediction_table(
            frame
        )

        if normalized is None:
            continue

        model_column = first_existing_column(
            normalized,
            (
                "run_id",
                "experiment_id",
                "model",
            ),
        )

        if model_column is not None:
            values = (
                normalized[model_column]
                .astype(str)
            )
            mask = (
                values.str.contains(
                    run_id,
                    regex=False,
                )
                | values.str.contains(
                    experiment_id,
                    regex=False,
                )
            )
            if mask.any():
                normalized = (
                    normalized.loc[mask]
                    .reset_index(drop=True)
                )

        if not normalized.empty:
            normalized.attrs[
                "source_path"
            ] = str(path)
            return normalized

    raise FileNotFoundError(
        "Не найдены test-предсказания "
        f"для {experiment_id}."
    )


try:
    cnn_predictions = load_predictions(
        selected["cnn"]["run_id"],
        selected["cnn"]["experiment_id"],
    )
    resnet_predictions = load_predictions(
        selected["resnet18"]["run_id"],
        selected["resnet18"][
            "experiment_id"
        ],
    )
except FileNotFoundError:
    run_test_evaluation()

    cnn_predictions = load_predictions(
        selected["cnn"]["run_id"],
        selected["cnn"]["experiment_id"],
    )
    resnet_predictions = load_predictions(
        selected["resnet18"]["run_id"],
        selected["resnet18"][
            "experiment_id"
        ],
    )

print(
    "CNN predictions:",
    cnn_predictions.attrs.get(
        "source_path"
    ),
)
print(
    "ResNet18 predictions:",
    resnet_predictions.attrs.get(
        "source_path"
    ),
)


### 5.4. Confusion matrix и метрики по классам


In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
)


def class_labels_from_predictions(
    *frames: pd.DataFrame,
) -> list:
    """Return a stable union of labels in prediction tables."""
    values = []

    for frame in frames:
        values.extend(
            frame["y_true"].tolist()
        )
        values.extend(
            frame["y_pred"].tolist()
        )

    return sorted(
        pd.unique(
            pd.Series(values)
        ).tolist(),
        key=lambda value: str(value),
    )


class_labels = class_labels_from_predictions(
    cnn_predictions,
    resnet_predictions,
)


def show_confusion_matrix(
    frame: pd.DataFrame,
    model_name: str,
) -> None:
    """Plot one confusion matrix."""
    matrix = confusion_matrix(
        frame["y_true"],
        frame["y_pred"],
        labels=class_labels,
    )

    display_object = ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=class_labels,
    )

    display_object.plot(
        xticks_rotation=45,
    )
    plt.title(
        f"Confusion matrix — {model_name}"
    )
    plt.tight_layout()
    plt.show()


show_confusion_matrix(
    cnn_predictions,
    "Custom CNN",
)

show_confusion_matrix(
    resnet_predictions,
    "ResNet18",
)


In [ ]:
def class_report(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return precision/recall/F1 by class."""
    report = classification_report(
        frame["y_true"],
        frame["y_pred"],
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )

    return (
        pd.DataFrame(report)
        .transpose()
        .round(4)
    )


print("Custom CNN — classification report")
display(
    class_report(cnn_predictions)
)

print("ResNet18 — classification report")
display(
    class_report(resnet_predictions)
)


### 5.5. Примеры правильных и ошибочных классификаций


In [ ]:
from PIL import Image


def resolve_image_path(
    value: str,
) -> Path | None:
    """Resolve an image path stored in a prediction artifact."""
    path = Path(str(value))

    candidates = [
        path,
        DATA / path,
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    matches = list(
        DATA.rglob(path.name)
    )

    if matches:
        return matches[0]

    return None


def sample_predictions(
    frame: pd.DataFrame,
    correct: bool,
    n_examples: int = 6,
) -> pd.DataFrame:
    """Choose representative correct or incorrect predictions."""
    mask = (
        frame["y_true"] == frame["y_pred"]
        if correct
        else frame["y_true"]
        != frame["y_pred"]
    )

    examples = frame.loc[mask].copy()

    if "confidence" in examples.columns:
        examples = examples.sort_values(
            "confidence",
            ascending=False,
        )

    return examples.head(
        n_examples
    )


def show_examples(
    frame: pd.DataFrame,
    model_name: str,
    correct: bool,
    n_examples: int = 6,
) -> None:
    """Display image examples or a table when paths are unavailable."""
    examples = sample_predictions(
        frame,
        correct=correct,
        n_examples=n_examples,
    )

    status = (
        "правильные"
        if correct
        else "ошибочные"
    )

    if examples.empty:
        print(
            f"{model_name}: "
            f"{status} примеры отсутствуют."
        )
        return

    if "image_path" not in examples.columns:
        columns = [
            column
            for column in (
                "y_true",
                "y_pred",
                "confidence",
            )
            if column in examples.columns
        ]
        print(
            f"{model_name}: {status} примеры"
        )
        display(
            examples[columns]
        )
        return

    resolved = []
    for row_index, row in examples.iterrows():
        image_path = resolve_image_path(
            row["image_path"]
        )
        if image_path is not None:
            resolved.append(
                (
                    row_index,
                    row,
                    image_path,
                )
            )

    if not resolved:
        columns = [
            column
            for column in (
                "image_path",
                "y_true",
                "y_pred",
                "confidence",
            )
            if column in examples.columns
        ]
        display(
            examples[columns]
        )
        return

    n_columns = 3
    n_rows = (
        len(resolved)
        + n_columns
        - 1
    ) // n_columns

    figure, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(12, 4 * n_rows),
        squeeze=False,
    )

    for axis, (_, row, image_path) in zip(
        axes.flat,
        resolved,
    ):
        with Image.open(
            image_path
        ) as image:
            axis.imshow(
                image.convert("RGB")
            )

        title = (
            f"Истина: {row['y_true']}\n"
            f"Прогноз: {row['y_pred']}"
        )

        if (
            "confidence" in row.index
            and pd.notna(
                row["confidence"]
            )
        ):
            title += (
                "\n"
                f"p={float(row['confidence']):.3f}"
            )

        axis.set_title(title)
        axis.axis("off")

    for axis in axes.flat[
        len(resolved):
    ]:
        axis.axis("off")

    figure.suptitle(
        f"{model_name}: "
        f"{status} классификации"
    )
    plt.tight_layout()
    plt.show()


show_examples(
    cnn_predictions,
    "Custom CNN",
    correct=False,
)
show_examples(
    cnn_predictions,
    "Custom CNN",
    correct=True,
)

show_examples(
    resnet_predictions,
    "ResNet18",
    correct=False,
)
show_examples(
    resnet_predictions,
    "ResNet18",
    correct=True,
)


## 6. Анализ результатов и выводы

Собственная CNN достигла на test accuracy **0.937667** и
macro-F1 **0.937451**. Основные ошибки связаны с визуально близкими
классами овощей.

ResNet18 достигла test accuracy **0.999000** и macro-F1 **0.998999**,
значительно превысив результат CNN с нуля.

Transfer learning оказался эффективнее обучения небольшой CNN с нуля
на данном наборе данных. Предобученная ResNet18 использует признаки,
полученные на ImageNet, а fine-tuning адаптирует их к 15 классам
целевого датасета.

Полученные результаты относятся к использованному датасету и принятому
протоколу эксперимента; качество на внешних данных требует отдельной
проверки.


## 7. Ответы на дополнительные вопросы

### 7.1. Архитектура ResNet18

ResNet18 — свёрточная нейронная сеть с остаточными блоками.
Shortcut-соединения передают вход блока напрямую к его выходу и
облегчают распространение градиента при обучении глубокой сети.

В работе финальный классификатор ResNet18 заменён на слой для 15 классов.

### 7.2. Принцип работы и параметры

Свёрточные слои извлекают локальные признаки изображения.
Residual-блоки последовательно формируют более сложные представления,
а shortcut-соединения стабилизируют обучение. После глобального
усреднения линейный слой формирует 15 логитов.

Сначала используется предобученная ResNet18, затем выполняется
fine-tuning с меньшим learning rate.

### 7.3. Отличие CNN от MLP

CNN сохраняет пространственную структуру изображения и использует
совместное использование весов: один фильтр применяется к различным
областям изображения.

В MLP изображение обычно разворачивается в вектор. Пространственная
структура явно не учитывается, а число параметров при работе с
изображениями может быть значительно больше.


## 8. Воспроизводимость

Для эксперимента зафиксированы:

- случайное зерно `seed = 42`;
- раздельные train, validation и test;
- конфигурации четырёх запусков;
- параметры оптимизатора и learning rate;
- сохранённые checkpoint и истории обучения;
- контрольные суммы исходного кода и используемых весов;
- параметры вычислительной среды.

Повторный запуск использует те же данные, конфигурации и начальные веса.


## 9. Источники

1. Vegetable Image Dataset, Kaggle — `misrakahmed/vegetable-image-dataset`.
2. PyTorch / torchvision — архитектура и предобученные веса ResNet18.
3. Материалы дисциплины «Прикладной искусственный интеллект».
